In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [73]:
df = pd.read_csv("train.txt", sep=';', header=None, names = ['text', 'emotion'])
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [74]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

labelling emotions

In [75]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i+=1

df['emotion'] = df['emotion'].map(emotion_numbers)
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


lowercasing

In [76]:
df['text'] = df['text'].apply(lambda x: x.lower())

remove punctuation

In [77]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

remove numbers

In [78]:
def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new +=i
    return new

df['text'] = df['text'].apply(remove_num)

remove emoji

In [79]:
def remove_emoji(txt):
    new = ''
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emoji)

remove stopwords

In [80]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [81]:
stop_words = set(stopwords.words("english"))

negation_words = {"not", "no", "never"}
stop_words = stop_words - negation_words


def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if i not in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove)

# MODELING FOR BoW

FEATURE EXTRACTION/VACTORIZATION(Bag of Words)

In [82]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer



vectorizer = CountVectorizer()
x = vectorizer.fit_transform(df["text"])
y = df['emotion']


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## base models for BoW

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [84]:
base_models_bow_dic = {'lr_base_model': LogisticRegression(), 'knn_base_model': KNeighborsClassifier(), 'svc_base_model': SVC(), 'nv_base_model': MultinomialNB(), 'dt_base_model': DecisionTreeClassifier(random_state=42), 'rf_base_model': RandomForestClassifier(random_state= 42, n_estimators=100)}
base_models_bow_accuracy = {}
base_models_bow_score = {}

In [85]:
for name, base_model_bow in base_models_bow_dic.items():
    base_model_bow.fit(x_train, y_train)
    y_predict = base_model_bow.predict(x_test)
    base_models_bow_accuracy[name] = accuracy_score(y_test, y_predict)
    base_models_bow_score[name] = base_model_bow.score(x_train, y_train)

In [86]:
base_models_bow_df = pd.DataFrame({
    'Accuracy': base_models_bow_accuracy,
    'Score': base_models_bow_score
}).sort_values(by= 'Accuracy', ascending=False)
base_models_bow_df

,Accuracy,Score
lr_base_model,0.888125,0.987266
rf_base_model,0.885312,0.997891
dt_base_model,0.869375,0.997891
svc_base_model,0.813438,0.973906
nv_base_model,0.770625,0.891719
knn_base_model,0.547188,0.724453


# MODELING FOR TF-IDF

FEATURE EXTRACTION/VACTORIZATION(TF-IDF)

In [87]:
vectorizer2 = TfidfVectorizer()
x = vectorizer2.fit_transform(df["text"])
y = df['emotion']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## base models for TF-IDF

In [88]:
base_models_tfidf_dic = {'lr_base_model': LogisticRegression(), 'knn_base_model': KNeighborsClassifier(), 'svc_base_model': SVC(), 'nv_base_model': MultinomialNB(), 'dt_base_model': DecisionTreeClassifier(random_state=42), 'rf_base_model': RandomForestClassifier(random_state= 42, n_estimators=100)}
base_models_tfidf_accuracy = {}
base_models_tfidf_score = {}

In [89]:
for name, base_model_tfidf in base_models_tfidf_dic.items():
    base_model_tfidf.fit(x_train, y_train)
    y_predict = base_model_tfidf.predict(x_test)
    base_models_tfidf_accuracy[name] = accuracy_score(y_test, y_predict)
    base_models_tfidf_score[name] = base_model_tfidf.score(x_train, y_train)

In [90]:
base_models_tfidf_df = pd.DataFrame({
    'Accuracy': base_models_tfidf_accuracy,
    'Score': base_models_tfidf_score
}).sort_values(by= 'Accuracy', ascending=False)
base_models_tfidf_df

,Accuracy,Score
rf_base_model,0.884375,0.997891
dt_base_model,0.860313,0.997891
lr_base_model,0.850313,0.946094
svc_base_model,0.837500,0.992031
knn_base_model,0.781250,0.868125
nv_base_model,0.654062,0.736641


In [91]:
base_models_bow_df = pd.DataFrame({
    'Accuracy': base_models_bow_accuracy,
    'Score': base_models_bow_score
}).sort_values(by= 'Accuracy', ascending=False)
base_models_bow_df

,Accuracy,Score
lr_base_model,0.888125,0.987266
rf_base_model,0.885312,0.997891
dt_base_model,0.869375,0.997891
svc_base_model,0.813438,0.973906
nv_base_model,0.770625,0.891719
knn_base_model,0.547188,0.724453


XGBOOST MODEL

In [92]:
from xgboost import XGBClassifier

xgboost_model = XGBClassifier(n_estimators = 100, max_depth = 10, learning_rate = 0.1, subsample = 0.8, colsample_bytree = 0.8)

xgboost_model.fit(x_train, y_train)
xgboost_model.predict(x_test)
xgboost_model_accuracy = accuracy_score(y_test, y_predict)
xgboost_model_score = xgboost_model.score(x_train, y_train)

In [93]:
print(f"Accuracy: {xgboost_model_accuracy}")
print(f"Score: {xgboost_model_score}")

Accuracy: 0.884375
Score: 0.9403125


hyper paraneter tuning random forest model

In [95]:
from sklearn.model_selection import RandomizedSearchCV


classifier = RandomizedSearchCV(
    estimator = RandomForestClassifier(),
    n_iter = 100,
    param_distributions= {'n_estimators': [100, 200], 'max_depth': [6, 10, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2], 'max_features': ['sqrt']},
    cv = 3,
    random_state = 42
)
classifier.fit(x_train, y_train)
rf_tuned_model = classifier.best_estimator_


In [96]:
y_predict = rf_tuned_model.predict(x_test)
rf_tuned_model_accuracy = accuracy_score(y_test, y_predict)
rf_tuned_model_score = rf_tuned_model.score(x_train, y_train)
rf_tuned_model_con_matrix = confusion_matrix(y_test, y_predict)

In [97]:
print(f"testing accuracy: {rf_tuned_model_accuracy}")
print(f"training score: {rf_tuned_model_score}")

testing accuracy: 0.8875
training score: 0.926640625


# FINAL MODEL: rf_tuned_model

In [98]:
import joblib

joblib.dump(rf_tuned_model, 'emotions_tuned_rf_model.pkl')
joblib.dump(vectorizer2, 'emotion_tfidf_vectorizer.pkl')

['emotion_tfidf_vectorizer.pkl']